In [ ]:
# Prompt Chaining Workflow
# This notebook demonstrates an advanced multi-step LLM workflow:
# 1. Generate an outline for a blog post
# 2. Create a full blog post from the outline
# 3. Summarize the generated blog post
#
# This showcases how to chain multiple LLM calls together using LangGraph

from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

# Initialize the LLM model
model = ChatOpenAI()

# Define the state schema for the prompt chaining workflow
class BlogState(TypedDict):
    """State for blog generation workflow
    
    Attributes:
        topic: The topic for which to generate a blog post
        outline: Generated outline for the blog post
        blog_post: Full blog post content
        summary: Summarized version of the blog post
    """
    topic: str
    outline: str
    blog_post: str
    summary: str

# Define workflow nodes for each step

def generate_outline(state: BlogState) -> BlogState:
    """Generate an outline for the blog post
    
    Args:
        state: Current workflow state containing the topic
        
    Returns:
        Updated state with the generated outline
    """
    prompt = f"""Generate a detailed outline for a blog post about: {state['topic']}
    
    Format the outline with numbered sections and subsections."""
    
    response = model.invoke(prompt)
    state['outline'] = response.content
    return state

def write_blog_post(state: BlogState) -> BlogState:
    """Write a full blog post based on the outline
    
    Args:
        state: Current workflow state containing the outline
        
    Returns:
        Updated state with the full blog post
    """
    prompt = f"""Based on the following outline, write a comprehensive blog post:
    
    Outline:
    {state['outline']}
    
    Write a detailed blog post with proper sections and paragraphs."""
    
    response = model.invoke(prompt)
    state['blog_post'] = response.content
    return state

def summarize_blog(state: BlogState) -> BlogState:
    """Summarize the generated blog post
    
    Args:
        state: Current workflow state containing the blog post
        
    Returns:
        Updated state with the summary
    """
    prompt = f"""Summarize the following blog post in 3-4 sentences:
    
    Blog Post:
    {state['blog_post']}
    
    Provide a concise summary that captures the main points."""
    
    response = model.invoke(prompt)
    state['summary'] = response.content
    return state

# Build the workflow graph
graph = StateGraph(BlogState)

# Add nodes for each workflow step
graph.add_node("generate_outline", generate_outline)
graph.add_node("write_blog_post", write_blog_post)
graph.add_node("summarize_blog", summarize_blog)

# Define the workflow flow
# The workflow executes in sequence: outline -> blog post -> summary
graph.add_edge(START, "generate_outline")
graph.add_edge("generate_outline", "write_blog_post")
graph.add_edge("write_blog_post", "summarize_blog")
graph.add_edge("summarize_blog", END)

# Compile the workflow
workflow = graph.compile()

# Example: Execute the workflow
# Uncomment to run:
# initial_state = {
#     "topic": "Introduction to LangGraph",
#     "outline": "",
#     "blog_post": "",
#     "summary": ""
# }
# 
# result = workflow.invoke(initial_state)
# print("OUTLINE:")
# print(result['outline'])
# print("\nBLOG POST:")
# print(result['blog_post'])
# print("\nSUMMARY:")
# print(result['summary'])